In [1]:
import pandas as pd
from pandas import DataFrame
from typing import Literal, List
import numpy as np

In [2]:
cleaned_dataset_address = "dataset/interim/past_dataset.csv"

In [ ]:
past_knowledge = pd.read_csv(cleaned_dataset_address, parse_dates=["datetime"], converters={"weather_code": str}).set_index("datetime")

In [4]:
past_knowledge.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5938 entries, 2005-01-01 to 2021-04-06
Data columns (total 33 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   general_dam_occupancy_rate                     5938 non-null   float64
 1   weather_code                                   5938 non-null   int64  
 2   temperature_2m_max                             5938 non-null   float64
 3   temperature_2m_min                             5938 non-null   float64
 4   temperature_2m_mean                            5938 non-null   float64
 5   apparent_temperature_max                       5938 non-null   float64
 6   apparent_temperature_min                       5938 non-null   float64
 7   apparent_temperature_mean                      5938 non-null   float64
 8   daylight_duration                              5938 non-null   float64
 9   sunshine_duration                 

In [5]:
class FeatureExtractor:
    def __init__(
        self,
        past_knowledge: DataFrame,
        cyclical_feature_names: List[str],
        freq: str = "D",
        lag_size: int = 30,
        window_size: int = 30,
    ):
        self.PAST_KNOWLEDGE = past_knowledge.sort_values(by="datetime")
        self.cyclical_feature_names = cyclical_feature_names
        self.lag_size = lag_size
        self.window_size = window_size
        self.freq = freq

    def transform(self, dates_to_predict: pd.DatetimeIndex) -> DataFrame:
        df = self._get_all_ranges(dates_to_predict)
        self.full_df = df.join(self.PAST_KNOWLEDGE, how="left")

        return (
            df.pipe(self._start_pipeline)
            .pipe(self._add_lag_features)
            .pipe(self._add_rolling_window_features)
            .pipe(self._add_exponential_moving_features)
            .pipe(self._drop_columns_with_same_values)
            .pipe(self._expand_datetime)
            .pipe(self._add_fourier_features)
            .pipe(
                lambda df: df.astype(
                    {
                        col: "int32"
                        for col in df.select_dtypes(["int", "uint32"]).columns
                    }
                )
            )
            .pipe(
                lambda df: df.astype(
                    {col: "float32" for col in df.select_dtypes("float").columns}
                )
            )
            .bfill()
            .loc[dates_to_predict, :]
        )

    def _start_pipeline(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.copy().sort_index()

    def _get_all_ranges(self, dates_to_predict: pd.DatetimeIndex) -> pd.DataFrame:
        start_date = min(dates_to_predict.min(), self.PAST_KNOWLEDGE.index.min())
        end_date = max(dates_to_predict.max(), self.PAST_KNOWLEDGE.index.max())
        complete_date_range = pd.date_range(
            start=start_date, end=end_date, freq=self.freq
        )
        return pd.DataFrame(index=complete_date_range)

    def _add_lag_features(
        self,
        df: DataFrame,
        fillna_with: Literal["ffill", "bfill"] | None = "bfill",
    ) -> DataFrame:
        columns_to_use = self.PAST_KNOWLEDGE.select_dtypes(
            include=["number", "object"]
        ).columns.tolist()

        created_features = [
            self.full_df[col].shift(i).rename(f"{col}_lag_{i}")
            for i in range(1, self.lag_size + 1)
            for col in columns_to_use
        ]

        lags_df = pd.concat(created_features, axis=1)

        df = df.join(
            lags_df,
            how="left",
        )

        if fillna_with == "ffill":
            df = df.ffill()
        elif fillna_with == "bfill":
            df = df.bfill()

        return df

    def _add_rolling_window_features(
        self,
        df: DataFrame,
        fillna_with: Literal["ffill", "bfill"] | None = "ffill",
    ) -> DataFrame:
        columns_to_use = self.PAST_KNOWLEDGE.select_dtypes(
            include=["float"]
        ).columns.tolist()

        metrics = ["mean", "std", "min", "max", "median", "var"]

        created_features = [
            (
                self.full_df[col]
                .rolling(window=size, min_periods=1)
                .agg(metrics)
                .rename(columns=lambda metric: f"{col}_rw{size}_{metric}")
            )
            for size in range(2, self.window_size + 1)
            for col in columns_to_use
        ]

        window_df = pd.concat(created_features, axis=1)

        df = df.join(
            window_df,
            how="left",
        )

        if fillna_with == "ffill":
            df = df.ffill()
        elif fillna_with == "bfill":
            df = df.bfill()

        return df

    def _drop_columns_with_same_values(self, df: DataFrame, threshold=0.9) -> DataFrame:
        to_drop = [
            col
            for col in df.columns
            if df[col].value_counts(normalize=True, dropna=False).values[0] >= threshold
        ]
        return df.drop(columns=to_drop)

    def _add_exponential_moving_features(
        self, df: pd.DataFrame, up_to: int = 30
    ) -> pd.DataFrame:
        columns_to_use = self.PAST_KNOWLEDGE.select_dtypes(
            include=["float"]
        ).columns.tolist()

        metrics = ["mean", "std", "var"]

        created_features = [
            (
                    self.full_df[col]
                    .ewm(span=span, adjust=False)
                    .agg(metrics)
                    .rename(columns=lambda metric: f"{col}_em_{span}_{metric}")
            )
            for span in range(2, up_to + 1)
            for col in columns_to_use
        ]

        exponential_moving_df = pd.concat(created_features, axis=1)
 
        df = df.join(
            exponential_moving_df,
            how="left",
        )
        return df

    def _expand_datetime(self, df: DataFrame) -> DataFrame:
        return df.assign(
            **{
                "year": lambda a_df: a_df.index.year,
                "month": lambda a_df: a_df.index.month,
                "day": lambda a_df: a_df.index.day,
                "hour": lambda a_df: a_df.index.hour,
                "day_of_year": lambda a_df: a_df.index.dayofyear,
                "week_of_year": lambda a_df: a_df.index.isocalendar().week,
                "quarter": lambda a_df: a_df.index.quarter,
                # "season": lambda a_df: a_df.index.month % 12 // 3 + 1,
                "is_weekend": lambda a_df: np.vectorize({True: 1, False: 0}.get)(a_df.index.weekday >= 5),
            }
        )

    def _add_fourier_features(self, df: pd.DataFrame, num_terms: int = 7) -> DataFrame:
        for col, max_val in self.cyclical_feature_names.items():
            source = self._get_column_source(df, col)

            for i in range(1, num_terms + 1):
                operation = 2 * np.pi * i * source[col] / max_val

                df[f"fourier_sin_{col}_{i}"] = np.sin(operation)
                df[f"fourier_cos_{col}_{i}"] = np.cos(operation)

        return df

    def _get_column_source(self, df: DataFrame, col: str) -> List[str]:
        if col in df.columns:
            source = df
        elif col in self.PAST_KNOWLEDGE.columns:
            source = self.PAST_KNOWLEDGE
        else:
            raise KeyError(f"{col} not found both in df and past knowledge.")
        return source


In [6]:
parameters = {
    "past_knowledge": past_knowledge,
    "cyclical_feature_names": {
        "month": 12,
        "day": 31,
        "day_of_year": 365,
        "week_of_year": 52,
        "quarter": 4,
        # "season": 4,
        "is_weekend": 2,
        "precipitation_hours": 24,
    },
    "lag_size": 30,
    "window_size": 30,
}

dates_to_predict = past_knowledge.index

In [7]:
feature_extractor = FeatureExtractor(**parameters)

In [8]:
prediction_df = feature_extractor.transform(dates_to_predict)

In [9]:
prediction_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5938 entries, 2005-01-01 to 2021-04-06
Columns: 9348 entries, general_dam_occupancy_rate_lag_1 to fourier_cos_precipitation_hours_7
dtypes: float32(9340), int32(8)
memory usage: 211.8 MB


In [11]:
prediction_df_null_counts = prediction_df.isna().sum().sort_values(ascending=False)
prediction_df_null_counts[prediction_df_null_counts > 0]

Series([], dtype: int64)

In [12]:
prediction_df.iloc[:, :40].head(5)

,general_dam_occupancy_rate_lag_1,weather_code_lag_1,temperature_2m_max_lag_1,temperature_2m_min_lag_1,temperature_2m_mean_lag_1,apparent_temperature_max_lag_1,apparent_temperature_min_lag_1,apparent_temperature_mean_lag_1,daylight_duration_lag_1,sunshine_duration_lag_1,...,climate_change_precipitation_sum_lag_1,climate_change_pressure_msl_mean_lag_1,climate_change_et0_fao_evapotranspiration_sum_lag_1,general_dam_occupancy_rate_lag_2,weather_code_lag_2,temperature_2m_max_lag_2,temperature_2m_min_lag_2,temperature_2m_mean_lag_2,apparent_temperature_max_lag_2,apparent_temperature_min_lag_2
datetime,,,,,,,,,,,,,,,,,,,,,
2005-01-01,44.619999,51.0,7.813000,4.713,6.660917,4.816814,2.181339,3.769840,33479.132812,22339.408203,...,0.127574,1015.549561,1.436427,44.619999,51.0,7.813,4.713,6.660917,4.816814,2.181339
2005-01-02,44.619999,51.0,7.813000,4.713,6.660917,4.816814,2.181339,3.769840,33479.132812,22339.408203,...,0.127574,1015.549561,1.436427,44.619999,51.0,7.813,4.713,6.660917,4.816814,2.181339
2005-01-03,44.619999,51.0,8.613000,2.263,6.031750,5.408062,-1.033089,3.009735,33529.406250,27602.402344,...,1.530689,1009.441345,0.990955,44.619999,51.0,7.813,4.713,6.660917,4.816814,2.181339
2005-01-04,44.470001,53.0,6.763000,2.463,5.388000,3.987309,-1.269303,1.578146,33583.265625,4671.306152,...,13.774413,1021.833252,1.098593,44.619999,51.0,8.613,2.263,6.031750,5.408062,-1.033089
2005-01-05,44.419998,51.0,9.662999,4.363,7.417167,6.653760,-0.225449,3.145541,33640.636719,25796.974609,...,0.127524,1025.425049,0.777874,44.470001,53.0,6.763,2.463,5.388000,3.987309,-1.269303
